# Предсказание цены квартиры

В этой тетрадке я обучаю CatBoost для предсказания цены и отдельно считаю примерный интервал цены. Интервал строится не просто как плюс-минус процент, а через две дополнительные модели.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Данные

Пока для модели используется уже очищенный датасет. Когда парсер ЦИАНа соберет достаточно объявлений, его можно будет привести к похожим колонкам и обучать модель уже на реальных объявлениях.

In [ ]:
df = pd.read_csv('../data/processed/flats_clean.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

## Небольшой анализ

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df['price'], bins=30)
plt.title('Распределение цены')
plt.xlabel('Цена')
plt.ylabel('Количество')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.heatmap(df.select_dtypes(include='number').corr(), annot=True, cmap='Blues')
plt.title('Корреляции числовых признаков')
plt.show()

## Подготовка признаков

In [ ]:
target = 'price'

features = [
    'area',
    'bedrooms',
    'bathrooms',
    'stories',
    'mainroad',
    'guestroom',
    'basement',
    'hotwaterheating',
    'airconditioning',
    'parking',
    'prefarea',
    'furnishingstatus'
]

cat_columns = [
    'mainroad',
    'guestroom',
    'basement',
    'hotwaterheating',
    'airconditioning',
    'prefarea',
    'furnishingstatus'
]

cat_features = [features.index(col) for col in cat_columns]

X = df[features]
y = df[target]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## Обучение CatBoost

Основная модель предсказывает саму цену. Еще две модели нужны для нижней и верхней границы интервала.

In [ ]:
params = {
    'iterations': 500,
    'learning_rate': 0.05,
    'depth': 6,
    'random_seed': 42,
    'train_dir': '../src/catboost_info',
    'verbose': 100
}

model = CatBoostRegressor(**params, loss_function='RMSE')
low_model = CatBoostRegressor(**params, loss_function='Quantile:alpha=0.1')
high_model = CatBoostRegressor(**params, loss_function='Quantile:alpha=0.9')

In [ ]:
model.fit(X_train, y_train, cat_features=cat_features)
low_model.fit(X_train, y_train, cat_features=cat_features)
high_model.fit(X_train, y_train, cat_features=cat_features)

## Метрики

In [ ]:
pred = model.predict(X_test)
low_pred = low_model.predict(X_test)
high_pred = high_model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)
mape = np.mean(np.abs((y_test - pred) / y_test)) * 100
median_error = np.median(np.abs(y_test - pred))

in_interval = ((y_test >= low_pred) & (y_test <= high_pred)).mean() * 100
average_width = np.mean(high_pred - low_pred)

print('MAE:', round(mae, 2))
print('RMSE:', round(rmse, 2))
print('R2:', round(r2, 3))
print('MAPE:', round(mape, 2), '%')
print('Median AE:', round(median_error, 2))
print('Попадание в интервал:', round(in_interval, 2), '%')
print('Средняя ширина интервала:', round(average_width, 2))

In [ ]:
results = pd.DataFrame({
    'real_price': y_test.values,
    'prediction': pred,
    'low': low_pred,
    'high': high_pred
})

results.head(10)

In [ ]:
plt.figure(figsize=(6, 6))
sns.scatterplot(x=results['real_price'], y=results['prediction'])
plt.xlabel('Реальная цена')
plt.ylabel('Предсказанная цена')
plt.title('Реальная и предсказанная цена')
plt.show()

## Сохранение моделей

In [ ]:
model.save_model('../models/catboost_model.cbm')
low_model.save_model('../models/catboost_low.cbm')
high_model.save_model('../models/catboost_high.cbm')

## Пример прогноза по введенным параметрам

Такую функцию потом сможет использовать телеграм-бот: он соберет ответы пользователя в словарь и передаст их в модель.

In [ ]:
def predict_price_from_params(flat):
    data = pd.DataFrame([flat])[features]

    price = model.predict(data)[0]
    low_price = low_model.predict(data)[0]
    high_price = high_model.predict(data)[0]

    return round(price), round(low_price), round(high_price)


example_flat = {
    'area': 6000,
    'bedrooms': 3,
    'bathrooms': 2,
    'stories': 2,
    'mainroad': 'yes',
    'guestroom': 'no',
    'basement': 'yes',
    'hotwaterheating': 'no',
    'airconditioning': 'yes',
    'parking': 1,
    'prefarea': 'yes',
    'furnishingstatus': 'semi-furnished',
}

price, low_price, high_price = predict_price_from_params(example_flat)

print('Предсказанная цена:', price)
print('Интервал:', low_price, '-', high_price)